# e3 · Effect of VIX, Elastic Net (block B3)

This notebook tests if VIX is doing most of the forecasting work, or if non-VIX variables also add information.
Pairs come from `research_config.BLOCKS["B3"]`.

**Tests:**
Each pair is nested (`a` contains `b`'s features), so MSE on log RV uses Clark-West. QLIKE uses Diebold-Mariano, which is undersized for nested models, so we only use their p-values as supporting evidence and read them with caution. For both, the loss differential is averaged across ETFs each month and tested with Newey-West (Andrews bandwidth). p-values are one-sided, H1: `a` beats `b`.

Note that Clark-West was derived for linear models estimated by OLS so applying to Elastic Net is an approximation. 

Output in `forecasting/results/`: `e3_losses.csv`, `e3_evaluation.csv`. `gain_t > 0` favours `a`. `loss_ratio` is `a`'s loss over `b`'s. `pct_change` is the ratio as a % change. the Clark-West statistic adjusts for nesting, so it can be significant while the ratio is above 1. We will primarily look at MSE on log RV with Clarke-West results.

In [ ]:
import sys
from pathlib import Path

_checkout = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "models" / "research_config.py").is_file()),
    None,
)
if _checkout is None:
    raise FileNotFoundError("Start the notebook kernel from within this project checkout.")
sys.path.insert(0, str(_checkout / "models"))
from utils.paths import PROJECT_ROOT, MODELS_DIR, DATASET_DIR, FORECAST_DIR, FORECAST_RESULTS_DIR

import sys
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import norm

from research_config import BLOCKS, TICKERS

FORECAST_PATH = MODELS_DIR / "forecasting" / "forecasts" / "en_forecasts.csv"
RESULTS_DIR = MODELS_DIR / "forecasting" / "results"
RESULTS_DIR.mkdir(exist_ok=True)

block = BLOCKS["B3"]
df = pd.read_csv(FORECAST_PATH, parse_dates=["date"])
available = set(df["model"])
pairs = [p for p in block["pairs"] if p[1] in available and p[2] in available]
print(f"{len(pairs)} of {len(block['pairs'])} B3 pairs runnable")

3 of 3 B3 pairs runnable


In [2]:
def andrews_bandwidth(x):
    x = np.asarray(x, float); n = len(x); xd = x - x.mean()
    den = float(xd[:-1] @ xd[:-1])
    rho = 0.0 if den <= 0 else float(xd[1:] @ xd[:-1]) / den
    rho = min(max(rho, -0.97), 0.97)
    a1 = 4.0 * rho ** 2 / ((1 - rho) ** 2 * (1 + rho) ** 2)
    return 0 if a1 <= 0 else int(min(np.floor(1.1447 * (a1 * n) ** (1 / 3)), n // 4))


def newey_west_lrv(x, lag):
    x = np.asarray(x, float); n = len(x); xd = x - x.mean()
    lag = max(0, min(lag, n - 1)); s = float(xd @ xd) / n
    for l in range(1, lag + 1):
        s += 2.0 * (1.0 - l / (lag + 1.0)) * float(xd[l:] @ xd[:-l]) / n
    return max(s, 1e-16)


def hac_t(d):
    """t-statistic of mean(d) with a Newey-West variance."""
    d = np.asarray(d, float)
    return d.mean() / np.sqrt(newey_west_lrv(d, andrews_bandwidth(d)) / len(d))


def qlike(rv, fc):
    r = rv / fc
    return r - np.log(r) - 1

## Losses

Common sample: the (ETF, month) rows where all four variants have a forecast.

In [3]:
models = [m for m in block["models"] if m in available]
d = df[df["model"].isin(models)].copy()
d = d[d.groupby(["etf", "date"])["model"].transform("nunique") == len(models)]
d["MSE_log"] = (d["actual_logRV"] - d["fc_log"]) ** 2
d["MSE_level"] = (d["actual_RV"] - d["fc_level"]) ** 2
d["QLIKE"] = qlike(d["actual_RV"], d["fc_level"])
print(f"{d['date'].min():%Y-%m} to {d['date'].max():%Y-%m}, {len(d)} rows")

LOSSES = ["QLIKE", "MSE_log", "MSE_level"]
per_etf = d.groupby(["model", "etf"])[LOSSES].mean().reset_index()
pooled = d.groupby("model")[LOSSES].mean().reset_index().assign(etf="pooled")
losses = pd.concat([per_etf, pooled], ignore_index=True).rename(columns={"etf": "ETF"})
losses.to_csv(RESULTS_DIR / "e3_losses.csv", index=False)
losses[losses["ETF"] == "pooled"].set_index("model")[LOSSES].round(4)

2008-02 to 2025-11, 4280 rows


,QLIKE,MSE_log,MSE_level
model,,,
en_full_macro,0.3366,0.4912,0.0002
en_no_macro,0.3279,0.5232,0.0001
en_no_vix,0.3938,0.5571,0.0002
en_vix_only,0.2953,0.4783,0.0013


## Tests

In [4]:
def one_sided_p(t):
    return float(1 - norm.cdf(t))


loss_idx = losses.set_index(["model", "ETF"])
rows = []
for pair_id, a, b, _ in pairs:
    A = d[d["model"] == a].set_index(["date", "etf"])
    B = d[d["model"] == b].set_index(["date", "etf"])
    dm_diff = A["QLIKE"] - B["QLIKE"]   # negative favours a
    # Clark-West adjusted differential; positive favours a
    cw = B["MSE_log"] - (A["MSE_log"] - (B["fc_log"] - A["fc_log"]) ** 2)

    for etf in ["pooled"] + TICKERS:
        if etf == "pooled":
            dm_t, cw_t = dm_diff.groupby("date").mean(), cw.groupby("date").mean()
        else:
            dm_t, cw_t = dm_diff.xs(etf, level="etf"), cw.xs(etf, level="etf")
        for loss, test, t in [("QLIKE", "DM", -hac_t(dm_t)), ("MSE_log", "Clark-West", hac_t(cw_t))]:
            rows.append({
                "pair": pair_id, "a": a, "b": b, "ETF": etf, "loss": loss, "test": test,
                "loss_ratio": loss_idx.loc[(a, etf), loss] / loss_idx.loc[(b, etf), loss],
                "gain_t": t, "p_value": one_sided_p(t),
            })

tests = pd.DataFrame(rows)
tests["pct_change"] = (tests["loss_ratio"] - 1) * 100   # negative: a has the lower loss
tests.to_csv(RESULTS_DIR / "e3_evaluation.csv", index=False)
tests[tests["ETF"] == "pooled"].drop(columns="ETF").round(4).reset_index(drop=True)

,pair,a,b,loss,test,loss_ratio,gain_t,p_value,pct_change
0,B3.1,en_full_macro,en_no_vix,QLIKE,DM,0.8549,1.7197,0.0427,-14.5137
1,B3.1,en_full_macro,en_no_vix,MSE_log,Clark-West,0.8817,3.6603,0.0001,-11.8291
2,B3.2,en_vix_only,en_no_macro,QLIKE,DM,0.9005,2.8487,0.0022,-9.9461
3,B3.2,en_vix_only,en_no_macro,MSE_log,Clark-West,0.9141,7.1218,0.0000,-8.5851
4,B3.3,en_full_macro,en_vix_only,QLIKE,DM,1.1401,-1.8112,0.9649,14.0059
5,B3.3,en_full_macro,en_vix_only,MSE_log,Clark-West,1.0271,1.2443,0.1067,2.7118


## By sector tests (exploratory)

In [5]:
SECTORS = {"XLB": "Materials", "XLE": "Energy", "XLF": "Financials", "XLI": "Industrials", "XLY": "Cons. Disc."}


def fmt_p(p):
    return "<0.001" if p < 0.001 else f"{p:.3f}"


def by_sector(loss):
    t = tests[tests["loss"] == loss].copy()
    t["cell"] = [f"{r:.2f} ({fmt_p(p)})" for r, p in zip(t["loss_ratio"], t["p_value"])]
    t["row"] = t["pair"] + ": " + t["a"] + " vs " + t["b"]
    table = t.pivot(index="row", columns="ETF", values="cell")[TICKERS + ["pooled"]]
    table.columns = [f"{c} {SECTORS[c]}" for c in TICKERS] + ["Pooled"]
    return table.rename_axis(None)


for loss, test in [("MSE_log", "Clark-West"), ("QLIKE", "DM")]:
    print(f"{loss}: ratio (p-value), {test}")
    display(by_sector(loss))

MSE_log: ratio (p-value), Clark-West


,XLB Materials,XLE Energy,XLF Financials,XLI Industrials,XLY Cons. Disc.,Pooled
B3.1: en_full_macro vs en_no_vix,0.90 (<0.001),0.98 (0.062),0.91 (<0.001),0.74 (0.008),0.89 (<0.001),0.88 (<0.001)
B3.2: en_vix_only vs en_no_macro,0.89 (<0.001),0.99 (0.041),0.85 (<0.001),0.87 (<0.001),1.01 (<0.001),0.91 (<0.001)
B3.3: en_full_macro vs en_vix_only,1.03 (0.244),1.21 (0.915),1.02 (0.056),0.96 (0.162),0.94 (0.032),1.03 (0.107)


QLIKE: ratio (p-value), DM


,XLB Materials,XLE Energy,XLF Financials,XLI Industrials,XLY Cons. Disc.,Pooled
B3.1: en_full_macro vs en_no_vix,0.89 (0.066),0.95 (0.119),0.89 (0.166),0.67 (0.031),0.90 (0.002),0.85 (0.043)
B3.2: en_vix_only vs en_no_macro,0.86 (<0.001),0.98 (0.279),0.87 (0.021),0.84 (<0.001),0.97 (0.265),0.90 (0.002)
B3.3: en_full_macro vs en_vix_only,1.11 (0.949),1.35 (0.969),1.14 (0.859),1.03 (0.702),1.06 (0.750),1.14 (0.965)


## By sector percentage changes

Percentage change in mean loss for `a` relative to `b`: `100 × (loss_ratio - 1)`.
Negative values mean lower loss for `a`; positive values mean higher loss.
The pooled row includes all five sectors.


In [6]:
for pair_id, a, b, _ in pairs:
    sector_pct_change = (
        tests.loc[tests["pair"].eq(pair_id)]
        .pivot(index="ETF", columns="loss", values="pct_change")
        .reindex(index=TICKERS + ["pooled"], columns=["MSE_log", "QLIKE"])
        .rename(columns={"MSE_log": "Log MSE"})
    )
    sector_pct_change.index = pd.Index(
        [f"{ticker} {SECTORS[ticker]}" for ticker in TICKERS] + ["Pooled"], name="Sector ETF"
    )
    sector_pct_change.columns.name = None
    print(f"{pair_id}: {a} relative to {b}")
    display(sector_pct_change.map(lambda value: f"{value:+.2f}%"))


B3.1: en_full_macro relative to en_no_vix


,Log MSE,QLIKE
Sector ETF,,
XLB Materials,-9.61%,-10.56%
XLE Energy,-1.77%,-5.21%
XLF Financials,-9.47%,-11.34%
XLI Industrials,-26.45%,-32.54%
XLY Cons. Disc.,-10.63%,-10.45%
Pooled,-11.83%,-14.51%


B3.2: en_vix_only relative to en_no_macro


,Log MSE,QLIKE
Sector ETF,,
XLB Materials,-11.37%,-13.93%
XLE Energy,-0.97%,-1.53%
XLF Financials,-15.23%,-13.21%
XLI Industrials,-13.38%,-15.76%
XLY Cons. Disc.,+0.92%,-3.26%
Pooled,-8.59%,-9.95%


B3.3: en_full_macro relative to en_vix_only


,Log MSE,QLIKE
Sector ETF,,
XLB Materials,+2.52%,+11.30%
XLE Energy,+20.69%,+34.60%
XLF Financials,+2.32%,+14.37%
XLI Industrials,-4.48%,+3.10%
XLY Cons. Disc.,-6.01%,+6.03%
Pooled,+2.71%,+14.01%
